# Ch4: DML and Merge with Deletion Vectors

### DML

#### INSERT

In [0]:
data=[(1,100,'a',1),(2,200,'b',1),(3,300,'c',1),(4,400,'d',1)]
df=spark.createDataFrame(data,["cust_id","income","name","tip"])

df.display()

cust_id,income,name,tip
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1


In [0]:
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/deltalakecatalog1/default/deltavol/dmlsink/")

In [0]:
data=[(5,100,'a',1),(6,200,'b',1),(7,300,'c',1),(8,400,'d',1)]
df=spark.createDataFrame(data,["cust_id","income","name","tip"])

df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/deltalakecatalog1/default/deltavol/dmlsink/")

In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

cust_id,income,name,tip
5,100,a,1
6,200,b,1
7,300,c,1
8,400,d,1
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1


#### Update

In [0]:
%sql
update delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`
set income=1000 
where cust_id=5

num_affected_rows
1


In [0]:
%sql
SELECT * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

cust_id,income,name,tip
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1
5,1000,a,1
6,200,b,1
7,300,c,1
8,400,d,1


### UPSERT

In [0]:
data=[(1,100,'xyz',1),(9,200,'b',1),(10,300,'c',1),(4,400,'d',4)]
df=spark.createDataFrame(data,["cust_id","income","name","tip"])

df.display()

cust_id,income,name,tip
1,100,xyz,1
9,200,b,1
10,300,c,1
4,400,d,4


In [0]:
from delta.tables import DeltaTable

In [0]:
dlt_obj=DeltaTable.forPath(spark,"/Volumes/deltalakecatalog1/default/deltavol/dmlsink/")

dlt_obj.alias("target").merge(df.alias("source"),"target.cust_id=source.cust_id")\
.whenMatchedUpdateAll()\
.whenNotMatchedInsertAll()\
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

cust_id,income,name,tip
2,200,b,1
3,300,c,1
5,1000,a,1
6,200,b,1
7,300,c,1
8,400,d,1
1,100,xyz,1
9,200,b,1
10,300,c,1
4,400,d,4


# Ch-5 Delta Log Level Schema Changes

In [0]:
data=[(1,100,'xyz',1),(9,200,'b',1),(10,300,'c',1),(4,400,'d',4)]
df=spark.createDataFrame(data,["cust_id","income","name","tip"])
df.display()

cust_id,income,name,tip
1,100,xyz,1
9,200,b,1
10,300,c,1
4,400,d,4


In [0]:
df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/deltalakecatalog1/default/deltavol/schemalevel/")

In [0]:
df=spark.read.format("parquet")\
    .load("/Volumes/deltalakecatalog1/default/deltavol/schemalevel/part-00000-7b0a388b-b701-46a2-9779-7c836328dd95.c000.snappy.parquet")

In [0]:
df.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4603388306301264>, line 1
----> 1 df.display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:100, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     96 def df_display(df, *args, **kwargs):
     97     """
     98     df.display() is an alias for display(df). Run help(display) for more information.
     99     """
--> 100     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_conne

In [0]:
%sql
ALTER TABLE delta.`/Volumes/deltalakecatalog1/default/deltavol/schemalevel/`
RENAME COLUMN name  TO customer_name

In [0]:
%sql
ALTER TABLE delta.`/Volumes/deltalakecatalog1/default/deltavol/schemalevel/`
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name',
                    'delta.minReaderVersion' = '2',
                    'delta.minWriterVersion' = '5');

ALTER TABLE delta.`/Volumes/deltalakecatalog1/default/deltavol/schemalevel/`
RENAME COLUMN name TO customer_name;

In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/schemalevel/`

cust_id,income,customer_name,tip
1,100,xyz,1
9,200,b,1
10,300,c,1
4,400,d,4


In [0]:
df=spark.read.format("json")\
    .load("/Volumes/deltalakecatalog1/default/deltavol/schemalevel/_delta_log/00000000000000000008.json")

In [0]:
df.display()

commitInfo,metaData
"List(0731-141929-3kao2abb-v2n, Databricks-Runtime/18.x-aarch64-photon-scala2.13, true, WriteSerializable, List(2955175781060921), RENAME COLUMN, List(customer_name, name), 81068ffb-a10f-4b30-91e8-e380d02fe0c3, 7, 1785510137175, 332fb61d-d0ff-4840-84e4-c5ddf93cabe2, 76548768579520, aishwaryasr.reddy@gmail.com)",null
null,"List(List(4, name, true), 1785507699883, List(parquet), bf9de151-e33a-4c8f-950f-8caea1bbc2b0, List(), {""type"":""struct"",""fields"":[{""name"":""cust_id"",""type"":""long"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":1,""delta.columnMapping.physicalName"":""cust_id""}},{""name"":""income"",""type"":""long"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":2,""delta.columnMapping.physicalName"":""income""}},{""name"":""customer_name"",""type"":""string"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":3,""delta.columnMapping.physicalName"":""name""}},{""name"":""tip"",""type"":""long"",""nullable"":true,""metadata"":{""delta.columnMapping.id"":4,""delta.columnMapping.physicalName"":""tip""}}]})"


# CH5 - Table Utility Commands

### **DESCRIBE Command**

In [0]:
%sql
describe deltalakecatalog1.default.first_table

col_name,data_type,comment
id_col1,bigint,null
salary,int,null
name,string,null


### **DETAIL Command**

In [0]:
%sql
describe detail deltalakecatalog1.default.first_table

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,99d04ff0-40e1-43c2-9b2c-45a495b19f7a,deltalakecatalog1.default.first_table,null,,2026-07-30T04:53:03.177Z,2026-07-30T04:53:05.000Z,List(),List(),0,0,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-a9622000-1d7f-448b-8169-f713f025a4f8, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-396af513-2fba-4dcd-9c0e-61db127edc96)",3,7,"List(appendOnly, deletionVectors, domainMetadata, identityColumns, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


### **EXTENDED Comamnd**

In [0]:
%sql
describe extended deltalakecatalog1.default.first_table

col_name,data_type,comment
id_col1,bigint,null
salary,int,null
name,string,null
,,
# Detailed Table Information,,
Catalog,deltalakecatalog1,
Database,default,
Table,first_table,
Created Time,Thu Jul 30 04:53:05 UTC 2026,
Last Access,UNKNOWN,


### **Data Versioning**

In [0]:
%sql
describe history delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-07-31T02:08:46.000Z,76548768579520,aishwaryasr.reddy@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2955175781060921),212e69d5-7564-4add-b5ad-bbe77938431e,0731-020001-g5934wwf-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 7361, p25FileSize -> 1645, numDeletionVectorsRemoved -> 1, minFileSize -> 1645, numAddedFiles -> 1, maxFileSize -> 1645, p75FileSize -> 1645, p50FileSize -> 1645, numAddedBytes -> 1645)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-07-31T02:08:43.000Z,76548768579520,aishwaryasr.reddy@gmail.com,MERGE,"Map(predicate -> [""(cust_id#11219L = cust_id#11223L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2955175781060921),212e69d5-7564-4add-b5ad-bbe77938431e,0731-020001-g5934wwf-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 5753, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 15231, materializeSourceTimeMs -> 1430, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 9535, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4099)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-07-30T17:02:44.000Z,76548768579520,aishwaryasr.reddy@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2955175781060921),d1751933-9e43-400f-af56-ffdbd7a85449,0730-165429-e8jb2e8v-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 4477, p25FileSize -> 1608, numDeletionVectorsRemoved -> 1, minFileSize -> 1608, numAddedFiles -> 1, maxFileSize -> 1608, p75FileSize -> 1608, p50FileSize -> 1608, numAddedBytes -> 1608)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-07-30T17:02:41.000Z,76548768579520,aishwaryasr.reddy@gmail.com,UPDATE,"Map(predicate -> [""(cust_id#11466L = 5)""])",null,List(2955175781060921),d1751933-9e43-400f-af56-ffdbd7a85449,0730-165429-e8jb2e8v-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4730, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2966, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1436, rewriteTimeMs -> 1728)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-30T16:58:00.000Z,76548768579520,aishwaryasr.reddy@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2955175781060921),b0f05eb7-a436-40f8-87b1-808a60efc284,0730-165429-e8jb2e8v-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1521)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-30T16:55:02.000Z,76548768579520,aishwaryasr.reddy@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2955175781060921),9925a7c1-7ba1-43cf-928e-9583e0d0f6c2,0730-165429-e8jb2e8v-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1520)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

cust_id,income,name,tip
2,200,b,1
3,300,c,1
5,1000,a,1
6,200,b,1
7,300,c,1
8,400,d,1
1,100,xyz,1
9,200,b,1
10,300,c,1
4,400,d,4


In [0]:
%sql
RESTORE delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` to version as of 3

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1608,1,1,1,1645,1608


In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

cust_id,income,name,tip
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1
5,1000,a,1
6,200,b,1
7,300,c,1
8,400,d,1


In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` version as of 2;

cust_id,income,name,tip
6,200,b,1
7,300,c,1
8,400,d,1
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1
5,1000,a,1


**Using Timestamp - Time Travel**

In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` timestamp  as of '2026-07-30T16:55:02.000+00:00';

cust_id,income,name,tip
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1


In [0]:
%sql
select * from delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` timestamp  as of  "2026-07-30T17:02:42.000+00:00";

cust_id,income,name,tip
6,200,b,1
7,300,c,1
8,400,d,1
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1
5,1000,a,1


### **TABLE PROPERTIES**

In [0]:
%sql
SHOW TBLPROPERTIES deltalakecatalog1.default.first_table

key,value
delta.enableDeletionVectors,true
delta.enableRowTracking,true
delta.feature.appendOnly,supported
delta.feature.deletionVectors,supported
delta.feature.domainMetadata,supported
delta.feature.identityColumns,supported
delta.feature.invariants,supported
delta.feature.rowTracking,supported
delta.minReaderVersion,3
delta.minWriterVersion,7


In [0]:
%sql
show tblproperties delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

key,value
delta.enableDeletionVectors,true
delta.feature.appendOnly,supported
delta.feature.deletionVectors,supported
delta.feature.invariants,supported
delta.minReaderVersion,3
delta.minWriterVersion,7


### **VACCUM Command**

In [0]:
%sql
VACUUM delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/`

path
dbfs:/Volumes/deltalakecatalog1/default/deltavol/dmlsink


In [0]:
%sql
VACUUM delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` RETAIN 0 HOURS

### **CLONING**

**DEEP CLONE**

In [0]:
%sql
create table if not exists deltalakecatalog1.default.clonetbl
clone delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` version as of 3

source_table_size,source_num_of_files,num_of_synced_transactions,num_removed_files,num_copied_files,removed_files_size,copied_files_size
1608,1,null,0,1,0,1608


In [0]:
%sql
select * from deltalakecatalog1.default.clonetbl

cust_id,income,name,tip
1,100,a,1
2,200,b,1
3,300,c,1
4,400,d,1
5,1000,a,1
6,200,b,1
7,300,c,1
8,400,d,1


**SHALLOW CLONE**

In [0]:
%sql
create table if not exists deltalakecatalog1.default.clonetblshallow
shallow clone delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` version as of 3

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7601205788311903>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'create table if not exists deltalakecatalog1.default.clonetblshallow\nshallow clone delta.`/Volumes/deltalakecatalog1/default/deltavol/dmlsink/` version as of 3\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_

In [0]:
%sql
create table if not exists deltalakecatalog1.default.clonetblshallow
shallow clone deltalakecatalog1.default.first_table 

In [0]:
%sql
select * from deltalakecatalog1.default.clonetblshallow

id_col1,salary,name
